# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/prarthanamahesh21-hub/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
# Check what is already available in the Week-5 notebook

print("Available variables:")
for name in [
    "features_w04",
    "model_frame",
    "model_df",
    "features_w05",
    "df",
    "data",
    "warehouse",
]:
    print(f"{name}: {name in globals()}")

Available variables:
features_w04: False
model_frame: False
model_df: False
features_w05: False
df: False
data: False
warehouse: False


In [4]:
# Show the currently defined DataFrame variables

import pandas as pd

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(name, "->", obj.shape, "->", obj.columns.tolist()[:10])

In [5]:
print("Setup complete.")

Setup complete.


In [6]:
import pandas as pd

print("DataFrames currently available:")

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(name, obj.shape)

DataFrames currently available:


In [12]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")

print("Hugging Face token loaded:", HF_TOKEN is not None)

Hugging Face token loaded: True


In [13]:
from datasets import load_dataset

dim_content = load_dataset(
    "FlyRank/internship-warehouse",
    "dim_content",
    token=HF_TOKEN
)

fact_query = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_query_90d",
    token=HF_TOKEN
)

print("Datasets loaded.")

README.md:   0%|          | 0.00/3.04k [00:00<?, ?B/s]

dim_content.parquet: reconstructing file:   0%|          |  0.00B / 19.6MB            

dim_content.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/519606 [00:00<?, ? examples/s]

fact_content_query_90d.parquet: reconstructing file:   0%|          |  0.00B / 60.7MB            

fact_content_query_90d.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/2414248 [00:00<?, ? examples/s]

Datasets loaded.


In [14]:
import duckdb

content = dim_content["train"]
query_90d = fact_query["train"]

con = duckdb.connect()

con.register("dim_content", content._data.table)
con.register("fact_content_query_90d", query_90d._data.table)

print("DuckDB setup complete")
print("Content rows:", len(content))
print("Query 90d rows:", len(query_90d))

DuckDB setup complete
Content rows: 519606
Query 90d rows: 2414248


In [15]:
features_w04 = con.sql("""
WITH query_by_content AS (
    SELECT
        content_hash_id,
        SUM(COALESCE(impressions_90d, 0)) AS gsc_impressions_90d,
        SUM(COALESCE(clicks_90d, 0)) AS gsc_clicks_90d,
        SUM(COALESCE(content_visible_query_count, 0)) AS visible_query_count
    FROM fact_content_query_90d
    GROUP BY content_hash_id
)

SELECT
    c.content_hash_id,

    DATE_DIFF(
        'day',
        CAST(c.content_created_date AS DATE),
        DATE '2026-03-01'
    ) AS content_age_days,

    c.word_count,

    COALESCE(q.gsc_impressions_90d, 0) AS gsc_impressions_90d,
    COALESCE(q.gsc_clicks_90d, 0) AS gsc_clicks_90d,
    COALESCE(q.visible_query_count, 0) AS visible_query_count

FROM dim_content c

LEFT JOIN query_by_content q
    ON c.content_hash_id = q.content_hash_id

WHERE c.content_created_date < DATE '2026-03-01'
  AND c.is_published IS TRUE
  AND c.is_deleted IS FALSE
""").df()

print("Feature frame shape:", features_w04.shape)
print(features_w04.head())

Feature frame shape: (303321, 6)
            content_hash_id  content_age_days  word_count  \
0  content_00014efc121d911d               214        <NA>   
1  content_000f4b73532b9e9d               213        <NA>   
2  content_001689f50648eef1               163        2464   
3  content_0026971d958baec2               163        <NA>   
4  content_002bf62039e49f93               214        <NA>   

   gsc_impressions_90d  gsc_clicks_90d  visible_query_count  
0                 70.0             1.0                  1.0  
1                 21.0             0.0                  1.0  
2                827.0             1.0                324.0  
3                 91.0             0.0                 16.0  
4                 88.0             0.0                 16.0  


In [16]:
print("Missing values:")
print(features_w04.isna().sum())

print("\nDuplicate content IDs:",
      features_w04["content_hash_id"].duplicated().sum())

print("\nRows:", len(features_w04))

Missing values:
content_hash_id             0
content_age_days            0
word_count             106638
gsc_impressions_90d         0
gsc_clicks_90d              0
visible_query_count         0
dtype: int64

Duplicate content IDs: 0

Rows: 303321


In [17]:
fact_performance = load_dataset(
    "FlyRank/internship-warehouse",
    "fact_content_daily_performance",
    token=HF_TOKEN
)

performance = fact_performance["train"]

con.register(
    "fact_content_daily_performance",
    performance._data.table
)

print("Performance table registered.")
print("Performance rows:", len(performance))

Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 19.6kB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 2.62MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 3.29MB            

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B / 7.12MB            

...

fact_content_daily_performance/month=202(…): reconstructing file:   0%|          |  0.00B /  146MB            

fact_content_daily_performance/month=202(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78835655 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/39 [00:00<?, ?it/s]

Performance table registered.
Performance rows: 78835655


In [19]:
feature_cols = [
    "content_age_days",
    "word_count",
    "gsc_impressions_90d",
    "gsc_clicks_90d",
    "ga4_pageviews_90d"
]

In [20]:
features_march = con.sql("""
WITH performance_90d AS (
    SELECT
        content_hash_id,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) AS gsc_impressions_90d,

        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_clicks, 0)
                ELSE 0
            END
        ) AS gsc_clicks_90d,

        SUM(
            CASE
                WHEN ga4_data_available IS TRUE
                THEN COALESCE(ga4_pageviews, 0)
                ELSE 0
            END
        ) AS ga4_pageviews_90d

    FROM fact_content_daily_performance

    WHERE report_date >= DATE '2025-12-01'
      AND report_date < DATE '2026-03-01'

    GROUP BY content_hash_id
)

SELECT
    c.content_hash_id,

    DATE_DIFF(
        'day',
        CAST(c.content_created_date AS DATE),
        DATE '2026-03-01'
    ) AS content_age_days,

    c.word_count,
    p.gsc_impressions_90d,
    p.gsc_clicks_90d,
    p.ga4_pageviews_90d

FROM dim_content c

LEFT JOIN performance_90d p
    ON c.content_hash_id = p.content_hash_id

WHERE c.content_created_date < DATE '2026-03-01'
  AND c.is_published IS TRUE
  AND c.is_deleted IS FALSE
""").df()

print("March feature frame:", features_march.shape)
print(features_march.columns.tolist())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

March feature frame: (303321, 6)
['content_hash_id', 'content_age_days', 'word_count', 'gsc_impressions_90d', 'gsc_clicks_90d', 'ga4_pageviews_90d']


In [22]:
future_label = con.sql("""
SELECT
    content_hash_id,
    SUM(
        CASE
            WHEN gsc_data_available IS TRUE
            THEN COALESCE(gsc_clicks, 0)
            ELSE 0
        END
    ) AS future_gsc_clicks

FROM fact_content_daily_performance

WHERE report_date >= DATE '2026-04-01'
  AND report_date < DATE '2026-05-01'

GROUP BY content_hash_id
""").df()

future_label["label"] = (
    future_label["future_gsc_clicks"] > 0
).astype(int)

print("Future label rows:", len(future_label))
print("Positive labels:", future_label["label"].sum())
print("Negative labels:", (future_label["label"] == 0).sum())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Future label rows: 362172
Positive labels: 67832
Negative labels: 294340


In [23]:
model_df = features_march.merge(
    future_label[["content_hash_id", "label"]],
    on="content_hash_id",
    how="inner"
)

print("Model frame shape:", model_df.shape)
print("Positive labels:", model_df["label"].sum())
print("Negative labels:", (model_df["label"] == 0).sum())

Model frame shape: (293504, 7)
Positive labels: 53259
Negative labels: 240245


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 1. Method choice and why

I use Logistic Regression as the first learned model for this lane. The task is to prioritize content pages for possible refresh, so a binary classification approach is appropriate for learning the relationship between the available page-level signals and the target outcome.

I chose Logistic Regression because it is relatively simple, interpretable, and provides a useful test of whether the audited signals provide additional predictive value when combined by a learned model. Starting with a simple model also makes the comparison with the Week-4 hand-written baseline easier to interpret.

The model uses the audited, non-leaky page-level features from the previous weeks. I will evaluate it using the same evaluation data and metric as the Week-4 baseline. The purpose is not to reward model complexity, but to determine whether the learned model provides measured improvement over the existing baseline.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

I use an 80/20 stratified train/test split with `random_state=42`. Each row represents one content page, so there is no repeated page-day sequence within this modeling frame that requires a time-series split. The outcome is future April performance, while all model inputs come from the March decision frame.

Stratification preserves the positive/negative label proportion in both sets. The test set is kept separate from model fitting and is also used to evaluate the Week-4 baseline, making the model-versus-baseline comparison like-for-like.


In [26]:
# Inspect the available modeling data before defining the split

print("features_w04 shape:", features_w04.shape)
print("features_w04 columns:")
print(features_w04.columns.tolist())

features_w04 shape: (303321, 6)
features_w04 columns:
['content_hash_id', 'content_age_days', 'word_count', 'gsc_impressions_90d', 'gsc_clicks_90d', 'visible_query_count']


In [24]:
from sklearn.model_selection import train_test_split

X = model_df[feature_cols]
y = model_df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Training positives:", y_train.sum())
print("Test positives:", y_test.sum())

Training rows: 234803
Test rows: 58701
Training positives: 42607
Test positives: 10652


In [25]:
# Check whether a target/label column is already present

possible_targets = [
    col for col in features_w04.columns
    if any(word in col.lower() for word in ["label", "target", "positive", "refresh", "action"])
]

print("Possible target columns:", possible_targets)

Possible target columns: []


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [27]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X_train, y_train)

model_scores = model.predict_proba(X_test)[:, 1]

model_auc = roc_auc_score(y_test, model_scores)

print("Logistic Regression ROC-AUC:", round(model_auc, 4))

Logistic Regression ROC-AUC: 0.8992


In [30]:
import numpy as np
import pandas as pd

print("NumPy and Pandas ready.")

NumPy and Pandas ready.


In [35]:
print("Test rows:", len(X_test))
print("Baseline test rows:", len(baseline_test))
print("Index aligned:", X_test.index.equals(baseline_test.index))

Test rows: 58701
Baseline test rows: 58701
Index aligned: True


In [33]:
# Evaluate the exact Week-4 baseline on the same test pages

baseline_test = features_w04.loc[X_test.index].copy()

baseline_test["age_points"] = pd.cut(
    baseline_test["content_age_days"],
    bins=[-1, 90, 180, 365, np.inf],
    labels=[0, 1, 2, 3]
).astype(int)

baseline_test["impression_points"] = pd.cut(
    baseline_test["gsc_impressions_90d"].fillna(0),
    bins=[-1, 0, 10, 100, 1000, np.inf],
    labels=[0, 1, 2, 3, 4]
).astype(int)

baseline_test["baseline_score"] = (
    baseline_test["age_points"]
    + baseline_test["impression_points"]
)

baseline_scores = baseline_test["baseline_score"]

baseline_auc = roc_auc_score(
    y_test,
    baseline_scores
)

print("Week-4 baseline ROC-AUC:", round(baseline_auc, 4))

Week-4 baseline ROC-AUC: 0.5308


In [34]:
comparison = pd.DataFrame({
    "method": [
        "Week-4 baseline",
        "Logistic Regression"
    ],
    "roc_auc": [
        baseline_auc,
        model_auc
    ]
})

comparison["roc_auc"] = comparison["roc_auc"].round(4)

comparison

,method,roc_auc
0,Week-4 baseline,0.5308
1,Logistic Regression,0.8992


In [36]:
print(comparison.to_string(index=False))

             method  roc_auc
    Week-4 baseline   0.5308
Logistic Regression   0.8992


In [37]:
improvement = model_auc - baseline_auc

print("Model ROC-AUC:", round(model_auc, 4))
print("Baseline ROC-AUC:", round(baseline_auc, 4))
print("Absolute improvement:", round(improvement, 4))

Model ROC-AUC: 0.8992
Baseline ROC-AUC: 0.5308
Absolute improvement: 0.3684


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 4. Errors and interpretation

The Logistic Regression model achieved a ROC-AUC of 0.8992 on the held-out test set, compared with 0.5308 for the Week-4 baseline. This is an absolute improvement of 0.3684 ROC-AUC.

The model combines the audited March signals rather than relying only on content age and impressions. Feature coefficients are inspected to understand which signals contribute most strongly to the model score.

Error inspection focuses on false positives and false negatives at a 0.5 probability threshold. A false positive is a page predicted to receive an April click when it did not, while a false negative is a page that received an April click but was assigned a probability below the threshold.

These errors are useful for understanding the limits of the model. A false positive may have strong March signals that did not translate into an April click, while a false negative may have relatively weak observed March signals but still received a click in April. These cases show that the available signals do not fully explain future page performance.

The ROC-AUC improvement is measured on the held-out test set and should be interpreted as predictive discrimination, not as evidence that the model causes better content performance.


In [38]:
# Inspect Logistic Regression coefficients

logreg = model.named_steps["logreg"]

coef_table = pd.DataFrame({
    "feature": feature_cols,
    "coefficient": logreg.coef_[0]
})

coef_table["abs_coefficient"] = coef_table["coefficient"].abs()

coef_table = coef_table.sort_values(
    "abs_coefficient",
    ascending=False
)

print("Feature coefficients:")
print(coef_table.to_string(index=False))

Feature coefficients:
            feature  coefficient  abs_coefficient
     gsc_clicks_90d     9.333780         9.333780
gsc_impressions_90d     2.728702         2.728702
  ga4_pageviews_90d    -1.371027         1.371027
   content_age_days    -0.495119         0.495119
         word_count     0.312918         0.312918


In [40]:
# Create a test-set prediction table for error inspection

error_df = X_test.copy()

error_df["actual"] = y_test.values
error_df["predicted_score"] = model_scores
error_df["predicted_class"] = (model_scores >= 0.5).astype(int)

error_df["error_type"] = np.select(
    [
        (error_df["actual"] == 1) & (error_df["predicted_class"] == 1),
        (error_df["actual"] == 0) & (error_df["predicted_class"] == 0),
        (error_df["actual"] == 0) & (error_df["predicted_class"] == 1),
        (error_df["actual"] == 1) & (error_df["predicted_class"] == 0)
    ],
    [
        "True Positive",
        "True Negative",
        "False Positive",
        "False Negative"
    ],
    default="Unknown"
)

print(error_df["error_type"].value_counts())

error_type
True Negative     47357
False Negative     5695
True Positive      4957
False Positive      692
Name: count, dtype: int64


In [41]:
print("Highest-confidence false positives:")
print(
    error_df[
        error_df["error_type"] == "False Positive"
    ]
    .sort_values("predicted_score", ascending=False)
    .head(10)
    .to_string()
)

print("\nHighest-confidence false negatives:")
print(
    error_df[
        error_df["error_type"] == "False Negative"
    ]
    .sort_values("predicted_score", ascending=True)
    .head(10)
    .to_string()
)

Highest-confidence false positives:
        content_age_days  word_count  gsc_impressions_90d  gsc_clicks_90d  ga4_pageviews_90d  actual  predicted_score  predicted_class      error_type
159261               380        <NA>             228819.0             2.0                0.0       0              1.0                1  False Positive
269817               380        <NA>             128101.0             0.0                0.0       0              1.0                1  False Positive
46610                186        6573              75230.0           784.0             1713.0       0              1.0                1  False Positive
19464                290        2972              88798.0            94.0              127.0       0              1.0                1  False Positive
196173               163        <NA>              63004.0            21.0                0.0       0              1.0                1  False Positive
60605                437        <NA>              52618.0 

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.